In [28]:
import math
import json
from pathlib import Path

import av
import numpy as np
import torch
import torch.nn.functional as F

from cosmos_tokenizer.video_lib import CausalVideoTokenizer

# ========= CONFIG =========
input_video = Path("/content/test-vid.mp4")
output_dir = Path("/content/1xgpt-test")
output_dir.mkdir(parents=True, exist_ok=True)

model_name = "Cosmos-Tokenizer-DV8x8x8"
encoder_path = Path("/content/pretrained_ckpts") / model_name / "encoder.jit"

rank = 0   # shard index, matching decoder convention
fps = 30   # expected frame rate
resolution = 256
frames_per_chunk = 17
# ==========================

# --- load encoder ---
encoder = CausalVideoTokenizer(checkpoint_enc=str(encoder_path))
if encoder._enc_model is None:
    raise RuntimeError(f"Failed to load encoder model from {encoder_path}")
print("Encoder initialized successfully.")

# --- video reader ---
container = av.open(str(input_video))
stream = container.streams.video[0]
stream.thread_type = "AUTO"

# --- read frames and convert to torch tensors ---
frames = []
for frame in container.decode(video=0):
    img = frame.to_ndarray(format="rgb24")              # (H,W,3), uint8
    tensor = torch.from_numpy(img).permute(2,0,1).float() / 255.0  # (3,H,W)
    tensor = F.interpolate(
        tensor.unsqueeze(0), size=(resolution, resolution),
        mode="bilinear", align_corners=False
    )  # (1,3,256,256)
    frames.append(tensor)

video_tensor = torch.cat(frames, dim=0)  # (T,3,256,256)
print(f"Loaded video tensor: {video_tensor.shape}")

# --- chunk into 17-frame segments ---
num_chunks = math.ceil(video_tensor.shape[0] / frames_per_chunk)
encoded_tokens = []

with torch.no_grad():
    for i in range(num_chunks):
        start = i * frames_per_chunk
        end = min((i + 1) * frames_per_chunk, video_tensor.shape[0])

        clip = video_tensor[start:end]  # (t,3,256,256)
        if clip.shape[0] < frames_per_chunk:
            # pad with last frame if not enough
            pad_frames = frames_per_chunk - clip.shape[0]
            pad = clip[-1:].repeat(pad_frames, 1, 1, 1)
            clip = torch.cat([clip, pad], dim=0)

        # permute to (C,T,H,W) then add batch: (1,3,17,256,256)
        clip = clip.permute(1,0,2,3).unsqueeze(0).cuda()

        # run encoder
        out = encoder.encode(clip)

        # handle tuple return
        if isinstance(out, (tuple, list)):
            tokens = out[0]
        else:
            tokens = out

        # convert to int32 for saving
        tokens = tokens.to(torch.int32).cpu().numpy()
        encoded_tokens.append(tokens[0])  # drop batch dim

        if i % 10 == 0:
            print(f"Processed chunk {i+1}/{num_chunks}")

encoded_tokens = np.stack(encoded_tokens, axis=0)  # (num_chunks,3,32,32)

# --- save binary shard ---
bin_path = output_dir / f"video_{rank}.bin"
encoded_video_dataset = np.memmap(
    bin_path, dtype=np.int32, mode="w+",
    shape=encoded_tokens.shape
)
encoded_video_dataset[:] = encoded_tokens[:]
encoded_video_dataset.flush()

# --- save metadata ---
metadata = {
    "shard_num_frames": video_tensor.shape[0],
    "fps": fps,
    "resolution": resolution,
    "frames_per_chunk": frames_per_chunk
}
metadata_path = output_dir / f"metadata_{rank}.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved encoded tokens to {bin_path}")
print(f"Saved metadata to {metadata_path}")




Encoder initialized successfully.
Loaded video tensor: torch.Size([304, 3, 256, 256])
Processed chunk 1/18
Processed chunk 11/18
Saved encoded tokens to /content/1xgpt-test/video_0.bin
Saved metadata to /content/1xgpt-test/metadata_0.json


In [4]:
!git clone https://github.com/NVIDIA/Cosmos-Tokenizer.git

Cloning into 'Cosmos-Tokenizer'...
remote: Enumerating objects: 197, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 197 (delta 37), reused 33 (delta 17), pack-reused 121 (from 1)
Receiving objects: 100% (197/197), 11.37 MiB | 30.10 MiB/s, done.
Resolving deltas: 100% (99/99), done.


In [5]:
# Step 2: # Install Cosmos-Tokenizer and its Python dependencies.
import os
if os.path.exists("Cosmos-Tokenizer"):
    os.chdir("Cosmos-Tokenizer")
    !apt-get update
    !apt-get install -y git-lfs
    !git lfs pull
    %pip install -e .
else:
    print('Cosmos-Tokenizer is already installed.')

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,008 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,798 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,274 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,327 kB]
Get:13 https://r2u.stat.illinois.edu/ubu

In [33]:
"""
NOTE: Download the Cosmos-Tokenizer repository and pre-trained model weights before running this script.
For full installation and setup instructions, please refer to:
https://github.com/NVIDIA/Cosmos-Tokenizer#readme
"""

import math
from pathlib import Path

import av
import numpy as np
import torch

from cosmos_tokenizer.utils import tensor2numpy
from cosmos_tokenizer.video_lib import CausalVideoTokenizer

input_dir = Path("/content/1xgpt-test")
output_dir = Path("/content/reconst_1xgpt/")
model_name = "Cosmos-Tokenizer-DV8x8x8"
decoder_path = Path("/content/pretrained_ckpts") / model_name / "decoder.jit"

print(f"Output directory exists: {input_dir.exists()}")
print(f"Decoder path exists: {decoder_path.exists()}")

rank = 0
metadata_path = input_dir / f"metadata_{rank}.json"
if not metadata_path.exists():
    raise FileNotFoundError(f"Metadata file not found at {metadata_path}")

with open(metadata_path, "r") as f:
    metadata_shard = json.load(f)

total_frames = metadata_shard["shard_num_frames"]
print(f"Total frames: {total_frames}")

encoded_video_dataset = np.memmap(input_dir / f"video_{rank}.bin", dtype=np.int32, mode="r", shape=(math.ceil(total_frames / 17), 3, 32, 32))

print(f"Encoded video dataset shape: {encoded_video_dataset.shape}")

indices = torch.tensor(encoded_video_dataset, device="cuda") if not isinstance(encoded_video_dataset, torch.Tensor) else encoded_video_dataset

try:
    decoder = CausalVideoTokenizer(checkpoint_dec=str(decoder_path))
    if decoder._dec_model is None:
        raise RuntimeError(f"Failed to load decoder model from {decoder_path}")
    print("Decoder initialized successfully.")
except Exception as e:
    raise RuntimeError(f"Error loading decoder: {str(e)}") from e

batch_size = 1
fps = 30
output_file = output_dir / "reconstructed_video.mp4"

first_batch = torch.from_numpy(encoded_video_dataset[0:1]).cuda()
with torch.no_grad():
    first_output = decoder.decode(first_batch).float()
    _, _, height, width = first_output.shape[-4:]

print(f"Output video dimensions: {width}x{height}")


ec = av.open(str(output_file), mode="w")
es = ec.add_stream("libx264", rate=30)   # H.264 CPU encoder
es.width = 256
es.height = 256


num_batches = math.ceil(len(encoded_video_dataset) / batch_size)
for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(encoded_video_dataset))

    batch = torch.from_numpy(encoded_video_dataset[start_idx:end_idx]).cuda()
    with torch.no_grad():
        # [B, 3, 17, 256, 256]
        reconstructed_batch = decoder.decode(batch)

    # (B, 17, 256, 256, 3)
    reconstructed_batch = tensor2numpy(reconstructed_batch)

    # frame: 17, 256, 256, 3
    for this_batch in reconstructed_batch:
        for single_frame in this_batch:  # Temporal dimension
            # 256, 256, 3
            for ep in es.encode(av.VideoFrame.from_ndarray(single_frame, format="rgb24")):
                ec.mux(ep)

    print(f"Processed batch {i + 1}/{num_batches}", flush=True)
    if i == 100:
        break

ec.close()
print(f"Video saved to: {output_file}")

Output directory exists: True
Decoder path exists: True
Total frames: 304
Encoded video dataset shape: (18, 3, 32, 32)
Decoder initialized successfully.
Output video dimensions: 256x256
Processed batch 1/18
Processed batch 2/18
Processed batch 3/18
Processed batch 4/18
Processed batch 5/18
Processed batch 6/18
Processed batch 7/18
Processed batch 8/18
Processed batch 9/18
Processed batch 10/18
Processed batch 11/18
Processed batch 12/18
Processed batch 13/18
Processed batch 14/18
Processed batch 15/18
Processed batch 16/18
Processed batch 17/18
Processed batch 18/18
Video saved to: /content/reconst_1xgpt/reconstructed_video.mp4
